In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, glob, os
import scipy.io as sio

## Objectives
1. Create preQC_df, do QC offline, load QC_df, store various cluster/unit/neuron figs in different directories
2. Create neur_df with columns for spikes, region, coordinates
3. Inspect & save

### variables

In [2]:
patient = 12
save = False

units_dir = f'../../data/units'
units_all_dir = f'{units_dir}/all/2025{patient}'
units_waves_dir = f'{units_dir}/waves/2025{patient}'
units_clean_dir = f'{units_dir}/clean/2025{patient}'
units_maybes_dir = f'{units_dir}/maybes/2025{patient}'
for dir in [units_all_dir, units_waves_dir, units_clean_dir, units_maybes_dir]:
    os.makedirs(dir, exist_ok=True)

## 1. QC stuff

In [3]:
save = True

# parse osort figs
for file in glob.glob(f'../../results/2025{patient}/osort_mat/figs/5/*'):

    # grab individual CL figs
    if 'CL' in os.path.basename(file) and 'ALL' not in os.path.basename(file):
        
        dest = os.path.join(units_all_dir, os.path.basename(file))
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {file} {units_all_dir}/')

    # grab WAVES figs to check for mergers
    if 'WAVES' in os.path.basename(file):
        
        dest = os.path.join(units_waves_dir, os.path.basename(file))
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {file} {units_waves_dir}/')

save = False # reset to be safe

### create preQC_df
note: clustID called unitID going forward

In [4]:
# init preQC_df 
preQC_df = pd.DataFrame(columns=['chanID', 'unitID'])
chanIDs, unitIDs = [], []

# parse units_all_dir
for file in glob.glob(f'{units_all_dir}/*'):
    chanIDs.append(int(os.path.basename(file).split('_')[0][1:]))
    unitIDs.append(int(os.path.basename(file).split('_')[2]))

# create df
preQC_df = pd.DataFrame({'chanID': chanIDs, 'unitID': unitIDs})
preQC_df = preQC_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)

# save df
preQC_df_path = f'../../results/2025{patient}/records/preQC_pt{patient}.csv'
if not os.path.exists(preQC_df_path):
    print(f'Saving preQC_df to {preQC_df_path}')
    preQC_df.to_csv(preQC_df_path, index=False)

save = False # reset to be safe
preQC_df

,chanID,unitID
0,97,596
1,97,612
2,98,1432
3,98,1502
4,98,1543
...,...,...
85,126,156
86,127,182
87,127,188
88,128,167


### load QC, and also save QC neur figs in units/clean/{patient} and units/maybes/{patient}

In [5]:
save=True

QC_df = pd.read_csv(f'../../results/2025{patient}/records/QC_pt{patient}.csv')
clean_df = QC_df[QC_df['keep'] == 1].copy().reset_index(drop=True)
maybe_df = QC_df[~QC_df['notes'].isna()].copy().reset_index(drop=True)

for unit_file in glob.glob(f'{units_all_dir}/*'):
    unitID = int(os.path.basename(unit_file).split('_')[2])
    
    if unitID in clean_df['unitID'].tolist():
        dest = f'{units_clean_dir}/{os.path.basename(unit_file)}'
        
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {unit_file} {dest}')
    
    if unitID in maybe_df['unitID'].tolist():
        dest = f'{units_maybes_dir}/{os.path.basename(unit_file)}'
        
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {unit_file} {dest}')

save = False
assert len(clean_df) == len(glob.glob(os.path.join(units_clean_dir, '*.png')))

save = False # reset to be safe
clean_df

AssertionError: 

## 2. neur_df stuff

### helpers

In [ ]:
def getunitID2spikes(unitIDs, spikes, clean_df):
    ''' return dict with keys=unique units, and vals = list of corresponding spikes '''
    
    # initialize unit2spikes with keys as unique unit IDs and empty list as values
    unit2spikes = {}
    for unitID, spike in zip(unitIDs, spikes):

        if unitID not in clean_df['unitID'].tolist(): continue
        if unitID not in unit2spikes: unit2spikes[unitID] = [] # initialize

        unit2spikes[unitID].append(spike)

    return unit2spikes

### First, add spike data from OSort mats.

In [ ]:
samp_rate = 1000000
# columns
chanID_list, unitID_list, spikes_list, num_spikes_list, FR_list = [], [], [], [], []

# go through OSort mat files
for mat_file in glob.glob(f'../../results/2025{patient}/osort_mat/sorted_mats/5/*_sorted_new.mat'):

    # load channel mat
    chanMat = sio.loadmat(mat_file)
    chanID = int(os.path.basename(mat_file).split('_')[0][1:])

    # load vectors of spike times and corresponding units
    if chanMat['assignedNegative'].size == 0: continue
    spike_vector = chanMat['newTimestampsNegative'][0] # len = total n_spikes
    unit_vector = chanMat['assignedNegative'][0] # len = total n_spikes

    # create unit_vector => [spike_vector] for QCed units
    unit2spikes = getunitID2spikes(unit_vector, spike_vector, clean_df)
    for unitID, unit_spike_list in unit2spikes.items():
        chanID_list.append(chanID)
        unitID_list.append(unitID)
        spikes = np.array(unit_spike_list) / samp_rate  # seconds
        spikes_list.append(spikes)
        num_spikes_list.append(len(spikes))
        FR_list.append(len(spikes) / (spikes[-1] - spikes[0]))

neur_df = pd.DataFrame({'chanID': chanID_list, 'unitID': unitID_list, 'spikes': spikes_list, 'num_spikes': num_spikes_list, 'FR': FR_list})
assert len(neur_df) == len(clean_df), f'Length mismatch: neur_df has {len(neur_df)} rows, clean_df has {len(clean_df)} rows'
neur_df


,chanID,unitID,spikes,num_spikes,FR
0,200,889,"[1.2737333333333334, 4.168333333333334, 5.4483...",3236,1.921668
1,201,2271,"[17.292166666666667, 17.33396666666667, 18.380...",4520,2.710302
2,201,2283,"[18.104666666666667, 25.1578, 32.396, 47.1109,...",2065,1.238960
3,193,929,"[1.4766333333333335, 1.5566666666666666, 1.716...",7246,4.303382
4,193,883,"[3.3423000000000003, 4.025933333333334, 4.1788...",2431,1.446764
5,204,1726,"[17.384533333333337, 17.67366666666667, 17.679...",2360,1.420181


### add region column by mapping channel -> region (label)

In [ ]:
chanMap = sio.loadmat(glob.glob(f'../../results/2025{patient}/records/*ChannelMap*.mat')[0])

# variable across patients
if patient in [12, 21]: channelMap = chanMap['ChannelMap1'].flatten()
elif patient == 18: channelMap = chanMap['ChannelMap2'].flatten()

labelMap = chanMap['LabelMap'].flatten() # contains region labels
labelMap = np.array([str(label.squeeze()) for label in labelMap])  # convert to str

nan_mask = ~np.isnan(channelMap)
channel2label = dict(zip(channelMap[nan_mask], labelMap[nan_mask]))

neur_df['region'] = neur_df['chanID'].map(channel2label).fillna(neur_df['chanID']).apply(lambda x: str(x))
neur_df

IndexError: boolean index did not match indexed array along axis 0; size of axis is 272 but size of corresponding boolean axis is 234

### add coordinates columns by mapping region->coords

In [ ]:
# cleaning function to handle nested arrays and bytes
def clean_entry(x):
    while isinstance(x, (np.ndarray, list)):
        x = x[0]
    if isinstance(x, (bytes, bytearray)):
        x = x.decode("utf-8", errors="ignore")
    return str(x)

at some point, figure out this code line by line

In [ ]:
# load
electrodeInfo = sio.loadmat(glob.glob(
                f'../../results/2025{patient}/records/*DI_Electrodes*.mat')[0])
ElecMapRaw   = pd.DataFrame(electrodeInfo['ElecMapRaw']) # region -> ID
ElecXYZRaw   = pd.DataFrame(electrodeInfo['ElecXYZRaw']) # ID -> coordinates
ElecAtlasRaw = pd.DataFrame(electrodeInfo['ElecAtlasRaw']) # atlas coords?

atlas_index = 0 # NMM

# clean
region_s = ElecMapRaw[0].apply(clean_entry)                    # Series of regions
atlas_s  = ElecAtlasRaw.iloc[:, atlas_index].apply(clean_entry)  # Series of atlas regions

# build small tables
region2id_df = pd.DataFrame({"region": region_s.values,
                             "ID": np.arange(len(region_s))})
id2xyz_df = ElecXYZRaw.reset_index().rename(columns={'index':'ID', 0:'x', 1:'y', 2:'z'})
# xyz2atlasRegions = pd.DataFrame({"ID": np.arange(len(atlas_s)),
#                                  "atlas_region": atlas_s.values})

# merge
final_neur_df = (neur_df
                .merge(region2id_df, on='region', how='left')
                .merge(id2xyz_df, on='ID', how='left')
                # .merge(xyz2atlasRegions, on='ID', how='left')
                )
final_neur_df = final_neur_df.drop(columns=['ID'])
final_neur_df['spikes'] = final_neur_df['spikes'].apply(lambda x: np.array(x))  # ensure arrays


### 3. inspect & save

In [ ]:
print('example neuron')
print(f'last 5 spikes (s): {final_neur_df["spikes"].iloc[0][-5:]}')
print(f'last 5 spikes (min): {final_neur_df["spikes"].iloc[0][-5:]/60}')

# add patient as first column
if 'patient' not in final_neur_df: final_neur_df.insert(0, 'patient', patient)

final_neur_df

example neuron
last 5 spikes (s): [1772.03653333 1772.04323333 1772.50243333 1773.21523333 1774.0054    ]
last 5 spikes (min): [29.53394222 29.53405389 29.54170722 29.55358722 29.56675667]


,patient,chanID,unitID,spikes,num_spikes,FR,region,x,y,z
0,21,218,3888,"[0.3418666666666667, 0.6712, 1.758166666666666...",3976,2.241688,mRAHIP7,18.600004,-1.29937,-15.600003
1,21,218,3887,"[0.34526666666666667, 0.3496, 0.35416666666666...",15705,8.859160,mRAHIP7,18.600004,-1.29937,-15.600003
2,21,222,878,"[0.05560000000000001, 0.7493333333333334, 0.82...",3460,1.949670,222,NaN,NaN,NaN
3,21,223,2489,"[0.047733333333333336, 0.1006, 0.1926, 0.2007,...",2159,1.216535,223,NaN,NaN,NaN
4,21,223,2492,"[2.8509333333333333, 3.1158333333333337, 3.505...",5723,3.261753,223,NaN,NaN,NaN
5,21,224,3097,"[5.0144, 5.237, 32.37466666666667, 34.28296666...",11870,6.796445,224,NaN,NaN,NaN


In [ ]:
# make dir
processed_data_dir = f'../../results/2025{patient}/records/processed_data'
os.makedirs(processed_data_dir, exist_ok=True)

# save file
parquet_path = os.path.join(processed_data_dir, 'df_neurs.parquet')
if os.path.exists(parquet_path):     print(f'File already exists, skipping: {parquet_path}')
else:     final_neur_df.to_parquet(parquet_path, index=False)
# final_neur_df.to_parquet(parquet_path, index=False)

File already exists, skipping: ../../results/202521/records/processed_data/df_neurs.parquet
